# NB00 — SPIRE data availability probe

**Goal:** Determine which data path is available and estimate the full scope of environmental bacterial MAGs with eggnog annotations.

**Data paths:**
1. `arkinlab.spire.eggnog_annotations_spire` — internal Databricks table (6,270 MAGs, `KEGG_ko` column).
2. **SPIRE download endpoints** (primary path for NB01):
   - `GET https://spire.embl.de/download_eggnog/{SAMPLE_ID}` — gzip eggnog TSV for all MAGs in a sample
   - `GET https://spire.embl.de/download_file/{MAG_ID}` — gzip FASTA with contigs for one MAG
   - These cover ALL 1.16M SPIRE MAGs, far beyond the internal table subset.

**Scope query:** Count all env bacterial MAGs (any coordinate coverage) via `refdata.spire.mag_coordinates + genome_metadata`.

**Output:** `data/spire_probe_results.json`

In [ ]:
print("NB00 executing — probing SPIRE data availability.")

In [ ]:
import json
import sys
from pathlib import Path

import pandas as pd

sys.path.insert(0, str(Path.cwd().parent / 'scripts'))

# Spark session — try berdl_notebook_utils (on-cluster), then repo script, then None
try:
    from berdl_notebook_utils.setup_spark_session import get_spark_session
    spark = get_spark_session()
    print("Spark connected via berdl_notebook_utils:", spark.version)
except Exception:
    try:
        repo_scripts = str(Path.cwd().parents[1] / 'scripts')
        sys.path.insert(0, repo_scripts)
        from get_spark_session import get_spark_session
        spark = get_spark_session()
        print("Spark connected via repo scripts:", spark.version)
    except Exception as e2:
        spark = None
        print(f"No Spark session available ({e2}) — will only test SPIRE API path.")

In [ ]:
# --------------------------------------------------------------------------
# Probe 1: arkinlab.spire.eggnog_annotations_spire
# --------------------------------------------------------------------------
internal_result = {"available": False, "n_env_mags": 0, "ko_coverage_pct": 0.0}

if spark is not None:
    try:
        df_ann = spark.sql("""
            SELECT
                COUNT(DISTINCT mag_id)                                              AS n_mags,
                SUM(CASE WHEN KEGG_ko != '-' AND KEGG_ko IS NOT NULL THEN 1 ELSE 0 END)
                    / COUNT(*) * 100                                                AS ko_coverage_pct
            FROM arkinlab.spire.eggnog_annotations_spire
            LIMIT 1
        """).toPandas()
        internal_result["available"] = True
        internal_result["n_mags_total"] = int(df_ann["n_mags"].iloc[0])
        internal_result["ko_coverage_pct"] = float(df_ann["ko_coverage_pct"].iloc[0])
        print("arkinlab.spire.eggnog_annotations_spire: accessible")
        print(df_ann)
    except Exception as e:
        print(f"arkinlab.spire.eggnog_annotations_spire not accessible: {e}")

print(internal_result)

In [ ]:
# Check environmental-sample MAG count (non-host ENVO terms)
# Join path: eggnog.mag_id -> mag_coordinates.mag_id (lat/lon)
#            eggnog.mag_id -> genome_metadata.genome_id (quality/domain)
#            mag_coordinates.sample_id -> sample_microntology.sample_id (env filter)
env_count_result = {"available": False, "n_env_mags": 0}

if spark is not None and internal_result["available"]:
    try:
        df_env = spark.sql("""
            SELECT COUNT(DISTINCT a.mag_id) AS n_env_mags
            FROM arkinlab.spire.eggnog_annotations_spire a
            JOIN refdata.spire.mag_coordinates mc ON a.mag_id = mc.mag_id
            JOIN refdata.spire.genome_metadata gm ON a.mag_id = gm.genome_id
            WHERE mc.latitude IS NOT NULL
              AND mc.longitude IS NOT NULL
              AND gm.domain = 'Bacteria'
              AND NOT EXISTS (
                SELECT 1 FROM refdata.spire.sample_microntology sm
                WHERE sm.sample_id = mc.sample_id
                  AND (sm.environment_term LIKE '%host%'
                    OR sm.environment_term LIKE '%gut%'
                    OR sm.environment_term LIKE '%clinical%')
              )
        """).toPandas()
        env_count_result["available"] = True
        env_count_result["n_env_mags"] = int(df_env["n_env_mags"].iloc[0])
        print("Environmental MAG count:", env_count_result["n_env_mags"])
    except Exception as e:
        print(f"env_count probe failed: {e}")

print(env_count_result)

In [ ]:
# --------------------------------------------------------------------------
# Probe 2: SPIRE download endpoints (download_eggnog and download_file)
# --------------------------------------------------------------------------
from spire_api import SPIREClient

client = SPIREClient(cache_dir=str(Path.cwd().parent / 'data' / 'spire_cache'))
api_probe = client.probe_endpoints()
print("SPIRE endpoint probe:")
for name, result in api_probe.items():
    print(f"  {name}: HTTP {result['status']}  ({result['url']})")

print()
print("Cache stats:", client.cache_stats())

In [ ]:
# --------------------------------------------------------------------------
# Probe 3: Full scope — count env bacterial MAGs across ALL of SPIRE
# (mag_coordinates has 1,148,021 rows; internal eggnog table has only 6,270)
# --------------------------------------------------------------------------
full_scope_result = {"available": False, "n_env_mags_full": 0, "n_samples_full": 0}

if spark is not None:
    try:
        df_scope = spark.sql("""
            SELECT
                COUNT(DISTINCT mc.mag_id)    AS n_env_mags,
                COUNT(DISTINCT mc.sample_id) AS n_samples
            FROM refdata.spire.mag_coordinates mc
            JOIN refdata.spire.genome_metadata gm ON mc.mag_id = gm.genome_id
            WHERE mc.latitude IS NOT NULL
              AND mc.longitude IS NOT NULL
              AND gm.domain = 'Bacteria'
              AND NOT EXISTS (
                SELECT 1 FROM refdata.spire.sample_microntology sm
                WHERE sm.sample_id = mc.sample_id
                  AND (sm.environment_term LIKE '%host%'
                    OR sm.environment_term LIKE '%gut%'
                    OR sm.environment_term LIKE '%clinical%')
              )
        """).toPandas()
        full_scope_result["available"] = True
        full_scope_result["n_env_mags_full"] = int(df_scope["n_env_mags"].iloc[0])
        full_scope_result["n_samples_full"] = int(df_scope["n_samples"].iloc[0])
        print(f"Full env bacterial MAG scope (mag_coordinates, no annotation filter):")
        print(f"  MAGs: {full_scope_result['n_env_mags_full']:,}")
        print(f"  Samples: {full_scope_result['n_samples_full']:,}")
        print(f"  Download estimate: ~{full_scope_result['n_samples_full'] * 36 // 1024} GB eggnog gzip")
    except Exception as e:
        print(f"Full scope query failed: {e}")

print(full_scope_result)

In [ ]:
# --------------------------------------------------------------------------
# Decision
# --------------------------------------------------------------------------
# NB01 will use SPIRE download endpoints to access ALL env bacterial MAGs,
# not just the 6,270 in the internal eggnog table.
# The internal table check below documents its status but does not gate NB01.
USE_INTERNAL_TABLE = (
    internal_result["available"]
    and env_count_result.get("n_env_mags", 0) >= 500
    and internal_result.get("ko_coverage_pct", 0) >= 50
)

decision = {
    "use_internal_table": USE_INTERNAL_TABLE,
    "use_spire_downloads": True,   # NB01 primary path: download_eggnog + download_file
    "use_spire_api": False,
    "internal_probe": internal_result,
    "env_count_internal": env_count_result,
    "env_count_full": full_scope_result,
    "endpoint_probe": api_probe,
}

out_path = Path.cwd().parent / 'data' / 'spire_probe_results.json'
out_path.parent.mkdir(parents=True, exist_ok=True)
with open(out_path, 'w') as f:
    json.dump(decision, f, indent=2)

print(f"Internal table: {'AVAILABLE' if USE_INTERNAL_TABLE else 'not usable'} ({env_count_result.get('n_env_mags', 0):,} env MAGs)")
print(f"Download endpoints: ENABLED  ({full_scope_result.get('n_env_mags_full', 0):,} env MAGs full scope)")
print(f"Saved to {out_path}")